# Log-return threshold forecast

Probability that the S&P 500 closes **above/below a level K** on the resolution date, estimated from the historical distribution of h-day log returns (h = trading days remaining).

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_market_calendars as mcal
from scipy.stats import norm

## Parameters

In [2]:
# Will the S&P 500 close above 7500 by the end of the year?
TICKER = "^GSPC"
K = 7500                      # threshold level
RESOLUTION_DATE = "2026-12-31"
DIRECTION = "above"           # "above" or "below"
VOL_WINDOW = 20               # trading days for realized vol
VOL_BAND = 0.25               # +/- relative band around today's vol for conditioning

## Data and horizon

In [3]:
px = yf.download(TICKER, start="1950-01-01", auto_adjust=False, progress=False)["Close"].squeeze().dropna()
S0 = float(px.iloc[-1])
last_close = px.index[-1]
logp = np.log(px)

sched = mcal.get_calendar("NYSE").schedule(start_date=last_close, end_date=RESOLUTION_DATE)
h = len(sched) - 1            # trading days after last_close, through resolution
assert h > 0, "Question already resolved"

d = np.log(K / S0)            # required log move

print(f"Last close {last_close.date()}  S0 = {S0:,.2f}")
print(f"Horizon h = {h} trading days   Required move d = {d:+.4f} ({np.exp(d)-1:+.2%})")

Last close 2026-11-13  S0 = 7,561.67
Horizon h = 32 trading days   Required move d = -0.0082 (-0.82%)


## Unconditional estimate

Fraction of all historical h-day windows whose log return crossed `d`, versus a normal fit to the same returns.

In [4]:
R = (logp.shift(-h) - logp).dropna()          # historical h-day log returns
hit = (R > d) if DIRECTION == "above" else (R < d)

p_emp = hit.mean()
z = (d - R.mean()) / R.std()
p_norm = 1 - norm.cdf(z) if DIRECTION == "above" else norm.cdf(z)

print(f"Windows: {len(R):,}   R mean = {R.mean():.4f}   R std = {R.std():.4f}")
print(f"Empirical: {p_emp:.1%}   Normal: {p_norm:.1%}")

for start in ["1950", "1970", "1990", "2010"]:
    sub = hit[start:]
    print(f"{start}+  {sub.mean():.1%}  (n = {sub.size:,})")

Windows: 19,308   R mean = 0.0101   R std = 0.0534
Empirical: 69.1%   Normal: 63.4%
1950+  69.1%  (n = 19,308)
1970+  68.8%  (n = 14,308)
1990+  71.4%  (n = 9,254)
2010+  75.1%  (n = 4,211)


## Conditioning on current volatility

Realized vol is computed from data up to *t* only, so there is no lookahead. Only windows that started at a similar vol to today are kept.

In [5]:
sigma = logp.diff().rolling(VOL_WINDOW).std() * np.sqrt(252)   # annualized
sigma_today = sigma.iloc[-1]

df = pd.DataFrame({"sigma": sigma, "R": R, "hit": hit.astype(float)}).dropna()
near = df[df.sigma.between(sigma_today * (1 - VOL_BAND), sigma_today * (1 + VOL_BAND))]

print(f"Sigma today: {sigma_today:.1%}   Mean sigma (history): {df.sigma.mean():.1%}")
print(f"Unconditional: {df.hit.mean():.1%}  (n = {len(df):,})")
print(f"Vol-matched:   {near.hit.mean():.1%}  (n = {len(near):,}, sigma within +/-{VOL_BAND:.0%} of today)")

Sigma today: 22.1%   Mean sigma (history): 13.5%
Unconditional: 69.1%  (n = 19,288)
Vol-matched:   71.1%  (n = 3,298, sigma within +/-25% of today)
